# WS-B — Weighted (covariate-shift) conformal for Moldova coverage

Replaces the **invalid** test-tuned ×1.10 width (which peeked at Moldova test labels) with
**weighted split-conformal** (Tibshirani et al. 2019): calibration residuals are reweighted by
the train→test likelihood ratio from a domain classifier over the frozen RAD-DINO features —
using **only unlabeled target features**, never test labels.

Runs Fusion + R6 for **Romania / Moldova / Kazakhstan** (5 seeds, M=10). The patched
`train_agentic.py` now reports `conformal_cov` **and** `weighted_conformal_cov` side by side.

**Attach** `mabdullahi454/tb-portals-cxr-pngs`, **GPU on**, then Run All. Download `ws_b_results.zip`.

## 0 — Clone branch `dl-sidework` (has the weighted-conformal patch) + install

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'dl-sidework'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'scikit-learn'], check=False)
print('ready')

## 1 — Paths + build the 5,010-image manifest

In [ ]:
import os, sys, pandas as pd
from pathlib import Path
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
FEATURES_CLS  = f'{WORK}/features_rad-dino_cls.npz'
PREDS_DIR = f'{WORK}/ws_b/preds'
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest size:', len(paper_df))

## 2 — Cache RAD-DINO CLS features (one-time; idempotent)

In [ ]:
from cache_features import main as cache_main
if os.path.isfile(FEATURES_CLS):
    print('cached ->', FEATURES_CLS)
else:
    cache_main(['--manifest', PAPER_MANIFEST, '--out', FEATURES_CLS,
                '--backbone', 'rad-dino', '--batch-size', '32'])

## 3 — Run Fusion + R6, 3 countries × 5 seeds, M=10
`results_agentic_fusion.csv` will carry `conformal_cov` (baseline) and `weighted_conformal_cov`.

In [ ]:
from src.training.train_agentic import main as agentic_main
os.makedirs(PREDS_DIR, exist_ok=True)
agentic_main(['--features', FEATURES_CLS, '--manifest', PAPER_MANIFEST,
              '--mode', 'fusion', '--rungs', '6', '--ensemble-m', '10',
              '--seeds', '0', '1', '2', '3', '4',
              '--held-outs', 'Romania', 'Moldova', 'Kazakhstan',
              '--out-dir', PREDS_DIR])
print('done')

## 4 — Baseline vs weighted conformal coverage (5-seed mean)

In [ ]:
import pandas as pd
res = pd.read_csv(f'{PREDS_DIR}/results_agentic_fusion.csv')
cols = ['conformal_cov', 'weighted_conformal_cov', 'conformal_width', 'weighted_conformal_width', 'weight_ess']
keep = res[res['rung'].isin(['rung6_conformal', 'agentic_best_tta'])]
table = (keep.groupby(['rung', 'held_out'])[cols].mean().reset_index()
             .rename(columns={'conformal_cov': 'cov_split', 'weighted_conformal_cov': 'cov_weighted',
                              'conformal_width': 'w_split', 'weighted_conformal_width': 'w_weighted'}))
table = table.round(3)
table.to_csv(f'{WORK}/ws_b/moldova_conformal_valid.csv', index=False)
print('Nominal target coverage = 0.90\n')
print(table.to_string(index=False))
print('\nMoldova split vs weighted:')
mol = table[(table.rung=='rung6_conformal') & (table.held_out=='Moldova')]
if len(mol):
    r = mol.iloc[0]
    print(f"  coverage {r.cov_split:.3f} -> {r.cov_weighted:.3f}   width {r.w_split:.1f} -> {r.w_weighted:.1f}   ess={r.weight_ess:.0f}")

## 5 — Bundle for download

In [ ]:
import shutil
print('preds  ->', shutil.make_archive(f'{WORK}/ws_b_preds', 'zip', PREDS_DIR))
print('results->', shutil.make_archive(f'{WORK}/ws_b_results', 'zip', f'{WORK}/ws_b'))